# 🛡️ AE-WGAN NSL-KDD — Version définitive (Table 5 exacte)

### Ce qui change :
- **Features ANOVA** : exactement les 24 de Table 5 du paper (hardcodées)
- **difficulty_level** gardé (= "Level" dans Table 5) — PAS supprimé
- **land** et **wrong_fragment** gardés — PAS supprimés comme quasi-constantes
- Architecture originale GitHub (BiGAN)
- Oversampling 10% de Normal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, urllib.request, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Preprocessing — Features exactes de Table 5

In [ ]:
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
for fname, furl in {"KDDTrain+.txt": url+"KDDTrain%2B.txt", "KDDTest+.txt": url+"KDDTest%2B.txt"}.items():
    if not os.path.exists(fname):
        urllib.request.urlretrieve(furl, fname)
        print(f"✅ {fname}")

NSL_KDD_COLUMNS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty_level'
]

NSL_KDD_ATTACK_MAP = {
    'normal':'Normal',
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS',
    'smurf':'DoS','teardrop':'DoS','mailbomb':'DoS','apache2':'DoS',
    'processtable':'DoS','udpstorm':'DoS',
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe',
    'mscan':'Probe','saint':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L',
    'multihop':'R2L','warezmaster':'R2L','warezclient':'R2L','spy':'R2L',
    'xlock':'R2L','xsnoop':'R2L','snmpguess':'R2L','snmpgetattack':'R2L',
    'httptunnel':'R2L','sendmail':'R2L','named':'R2L','worm':'R2L',
    'buffer_overflow':'U2R','loadmodule':'U2R','rootkit':'U2R',
    'perl':'U2R','sqlattack':'U2R','xterm':'U2R','ps':'U2R',
}

# ============================================================
# PAPER TABLE 5: exactement ces 24 features après ANOVA
# NOTE: "Level" = difficulty_level, on le garde!
# NOTE: land et wrong_fragment ne sont PAS supprimés
# ============================================================
PAPER_24_FEATURES = [
    'protocol_type', 'flag', 'land', 'wrong_fragment', 'logged_in',
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_srv_rerror_rate', 'difficulty_level'
]

# Charger
df_train = pd.read_csv("KDDTrain+.txt", header=None, names=NSL_KDD_COLUMNS)
df_test = pd.read_csv("KDDTest+.txt", header=None, names=NSL_KDD_COLUMNS)
df_train['label'] = df_train['label'].str.strip().str.rstrip('.')
df_test['label'] = df_test['label'].str.strip().str.rstrip('.')

# Labels
y_train_cat = df_train['label'].map(NSL_KDD_ATTACK_MAP).fillna('Unknown')
y_test_cat = df_test['label'].map(NSL_KDD_ATTACK_MAP).fillna('Unknown')
y_train_detail = df_train['label'].copy()

# Sélectionner exactement les 24 features du paper
X_train_df = df_train[PAPER_24_FEATURES].copy()
X_test_df = df_test[PAPER_24_FEATURES].copy()

print(f"Features sélectionnées (Table 5): {list(X_train_df.columns)}")
print(f"Nombre: {len(X_train_df.columns)}")

# Ordinal Encoding pour les catégorielles
cat_cols = X_train_df.select_dtypes(include=['object']).columns.tolist()
print(f"Features catégorielles à encoder: {cat_cols}")
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_df[cat_cols] = enc.fit_transform(X_train_df[cat_cols])
X_test_df[cat_cols] = enc.transform(X_test_df[cat_cols])

# Min-Max Scaling
scaler = MinMaxScaler()
X_train_final = scaler.fit_transform(X_train_df.values)
X_test_final = scaler.transform(X_test_df.values)

INPUT_DIM = 24
N_NORMAL = (y_train_cat == 'Normal').sum()
TARGET_PER_TYPE = int(N_NORMAL * 0.10)

print(f"\n✅ Preprocessing terminé (features EXACTES de Table 5)")
print(f"   Train: {X_train_final.shape}, Test: {X_test_final.shape}")
print(f"   Normal: {N_NORMAL}, Seuil 10%: {TARGET_PER_TYPE}")
print(f"\nDistribution:")
print(y_train_cat.value_counts())

## 2. AE-WGAN (architecture GitHub)

In [ ]:
class AEWGAN_Original:
    def __init__(self):
        self.img_rows = 4
        self.img_cols = 6
        self.channels = 1
        self.img_shape = (self.img_rows, self.img_cols, self.channels)
        self.latent_dim = 32
        self.flat_dim = int(np.prod(self.img_shape))
        optimizer = tf.keras.optimizers.Adamax(0.00009)
        self.discriminator = self._build_discriminator()
        self.discriminator.compile(loss=self._wasserstein_loss, optimizer=optimizer, metrics=['accuracy'])
        self.generator = self._build_generator()
        self.encoder = self._build_encoder()
        self.discriminator.trainable = False
        z = Input(shape=(self.latent_dim,))
        img_ = self.generator(z)
        img = Input(shape=self.img_shape)
        z_ = self.encoder(img)
        fake = self.discriminator([z, img_])
        valid = self.discriminator([z_, img])
        self.combined = Model([z, img], [fake, valid])
        self.combined.compile(loss=[self._wasserstein_loss, self._wasserstein_loss], optimizer=optimizer)

    def _wasserstein_loss(self, y_true, y_pred):
        return K.mean(y_true * y_pred)

    def _build_encoder(self):
        model = Sequential([Flatten(input_shape=self.img_shape),
                             Dense(20, activation='relu'), Dense(self.latent_dim)])
        img = Input(shape=self.img_shape)
        return Model(img, model(img), name='encoder')

    def _build_generator(self):
        model = Sequential([
            Dense(20, input_dim=self.latent_dim, activation='relu'),
            Dense(40, activation='relu'), Dense(80, activation='relu'),
            Dense(120, activation='relu'), Dense(160, activation='relu'),
            Dense(self.flat_dim, activation='relu'), Reshape(self.img_shape)])
        z = Input(shape=(self.latent_dim,))
        return Model(z, model(z), name='generator')

    def _build_discriminator(self):
        z = Input(shape=(self.latent_dim,))
        img = Input(shape=self.img_shape)
        d_in = Concatenate()([z, Flatten()(img)])
        x = Dense(80, activation='relu')(d_in)
        x = Dense(40, activation='relu')(x)
        validity = Dense(1, activation='sigmoid')(x)
        return Model([z, img], validity, name='discriminator')

    def train(self, X_attack_2d, epochs=101, batch_size=32, verbose=1):
        reshaped = X_attack_2d.reshape(-1, self.img_rows, self.img_cols, self.channels)
        n_samples = reshaped.shape[0]
        if n_samples < batch_size:
            batch_size = max(n_samples // 2, 2)
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))
        for epoch in range(epochs):
            z = np.random.normal(0, 1, size=(batch_size, self.latent_dim))
            imgs_ = self.generator.predict(z, verbose=0)
            idx = np.random.randint(0, n_samples, batch_size)
            imgs = reshaped[idx]
            z_ = self.encoder.predict(imgs, verbose=0)
            d_loss_real = self.discriminator.train_on_batch([z_, imgs], valid)
            d_loss_fake = self.discriminator.train_on_batch([z, imgs_], fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
            g_loss = self.combined.train_on_batch([z, imgs], [valid, fake])
            if verbose and (epoch+1) % max(1, epochs//5) == 0:
                print(f"      Epoch {epoch+1}/{epochs} — D loss: {d_loss[0]:.4f}, D acc: {100*d_loss[1]:.1f}%, G loss: {g_loss[0]:.4f}")

    def generate_batch(self, n):
        noise = np.random.normal(0, 1, size=(n, self.latent_dim))
        generated = self.generator.predict(noise, verbose=0, batch_size=256)
        return generated.reshape(-1, self.flat_dim)

print("✅ AEWGAN_Original défini")

## 3. Oversampling — 10% de Normal

In [ ]:
print("=" * 60)
print(f"AE-WGAN OVERSAMPLING — seuil: {TARGET_PER_TYPE} samples")
print("=" * 60)

attack_types = sorted([t for t in y_train_detail.unique() if t != 'normal'])
MIN_SAMPLES = 10

all_synthetic_X = []
all_synthetic_y = []
trained_models = {}
start_time = time.time()

for attack_name in attack_types:
    mask = y_train_detail == attack_name
    X_attack = X_train_final[mask]
    n_orig = X_attack.shape[0]

    if n_orig >= TARGET_PER_TYPE:
        print(f"  [{attack_name}] ⏩ {n_orig} >= {TARGET_PER_TYPE} — skip")
        continue
    if n_orig < MIN_SAMPLES:
        print(f"  [{attack_name}] ⏭️ {n_orig} < {MIN_SAMPLES} — skip")
        continue

    n_to_gen = TARGET_PER_TYPE - n_orig
    print(f"\n  [{attack_name}] 🧠 Training ({n_orig} samples, +{n_to_gen} synthétiques)...")

    model = AEWGAN_Original()
    bs = min(32, max(n_orig // 2, 2))
    model.train(X_attack, epochs=101, batch_size=bs, verbose=1)
    synthetic = model.generate_batch(n_to_gen)
    all_synthetic_X.append(synthetic)
    all_synthetic_y.extend([attack_name] * n_to_gen)
    trained_models[attack_name] = model
    print(f"  [{attack_name}] ✅ {n_orig} + {n_to_gen} = {n_orig + n_to_gen}")

elapsed = (time.time() - start_time) / 60
print(f"\n--- {elapsed:.1f} minutes ---")

if all_synthetic_X:
    X_synthetic_all = np.vstack(all_synthetic_X)
    y_synthetic_all = pd.Series(all_synthetic_y)
    X_augmented = np.vstack([X_train_final, X_synthetic_all])
    y_augmented_detail = pd.concat([y_train_detail, y_synthetic_all], ignore_index=True)
    y_augmented_cat = y_augmented_detail.map(NSL_KDD_ATTACK_MAP).fillna('Unknown')
else:
    X_augmented = X_train_final
    y_augmented_cat = y_train_cat

print(f"\n  Original:    {X_train_final.shape[0]:>10}")
print(f"  Synthétique: {X_augmented.shape[0]-X_train_final.shape[0]:>10}")
print(f"  Total:       {X_augmented.shape[0]:>10}")

## 4. Qualité générative

In [ ]:
def discrimination_test(X_real, X_synth):
    n = min(len(X_real), len(X_synth), 5000)
    X_r = X_real[np.random.choice(len(X_real), n, replace=n>len(X_real))]
    X_s = X_synth[np.random.choice(len(X_synth), n, replace=n>len(X_synth))]
    X_c = np.vstack([X_r, X_s]); y_c = np.array([1]*n + [0]*n)
    X_tr,X_te,y_tr,y_te = train_test_split(X_c,y_c,test_size=0.3,random_state=42,stratify=y_c)
    clf = RandomForestClassifier(100, random_state=42, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

print(f"{'─'*55}")
print(f"  {'Attaque':<18} {'Orig':>6} {'Gen':>6} {'DiscAcc':>8} {'Qualité':>9}")
print(f"{'─'*55}")
for name, model in trained_models.items():
    mask = y_train_detail == name
    X_real = X_train_final[mask]
    X_synth = model.generate_batch(min(len(X_real)*2, 1000))
    da = discrimination_test(X_real, X_synth)
    q = "Excellent" if da<0.55 else "Good" if da<0.65 else "Fair" if da<0.75 else "Poor"
    print(f"  {name:<18} {len(X_real):>6} {TARGET_PER_TYPE-len(X_real):>6} {da:>7.2%} {q:>9}")
print(f"{'─'*55}")

## 5. Classifieurs

In [ ]:
def build_dt(nc=2):
    return DecisionTreeClassifier(max_depth=7 if nc==2 else 9, random_state=42)
def build_rf(nc=2):
    return RandomForestClassifier(100, max_depth=8 if nc==2 else 10, random_state=42, n_jobs=-1)
def build_cnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential([layers.Input(shape=(dim,1)),layers.Conv1D(40,3,activation='relu',padding='same'),
        layers.Dropout(0.1),layers.Flatten(),layers.Dense(40,activation='relu'),
        layers.Dense(out,activation=act)],name='CNN')
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
def build_gru(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    if nc==2:
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10,activation='relu'));m.add(layers.Dense(10,activation='relu'))
    else:
        m.add(layers.GRU(30,activation='relu',return_sequences=True));m.add(layers.Dropout(0.1))
        m.add(layers.GRU(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(30,activation='relu'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
def build_lstm(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='LSTM'); m.add(layers.Input(shape=(dim,1)))
    if nc==2:
        m.add(layers.LSTM(10,activation='relu',return_sequences=True))
        m.add(layers.LSTM(20,activation='relu'));m.add(layers.Dropout(0.5))
        m.add(layers.Dense(10,activation='relu'))
    else:
        m.add(layers.LSTM(10,activation='relu',return_sequences=True));m.add(layers.Dropout(0.1))
        m.add(layers.LSTM(20,activation='relu',return_sequences=True));m.add(layers.Dropout(0.2))
        m.add(layers.LSTM(20,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10,activation='relu'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
def build_rnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='RNN'); m.add(layers.Input(shape=(dim,1)))
    if nc==2:
        m.add(layers.SimpleRNN(30,activation='relu'));m.add(layers.Dropout(0.1))
        m.add(layers.Dense(10,activation='relu'));m.add(layers.Dense(10,activation='relu'))
    else:
        m.add(layers.SimpleRNN(50,activation='relu',return_sequences=True));m.add(layers.Dropout(0.2))
        m.add(layers.SimpleRNN(50,activation='relu',return_sequences=True));m.add(layers.Dropout(0.2))
        m.add(layers.SimpleRNN(50,activation='relu',return_sequences=True));m.add(layers.Dropout(0.2))
        m.add(layers.SimpleRNN(50,activation='relu'));m.add(layers.Dropout(0.2))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
print("✅ Classifieurs définis")

## 6. Classification

In [ ]:
def prepare_labels(y_train,y_test,binary=True):
    le=LabelEncoder()
    if binary: return (y_train!='Normal').astype(int).values,(y_test!='Normal').astype(int).values,le
    else:
        le.fit(list(set(y_train.unique())|set(y_test.unique())))
        return le.transform(y_train),le.transform(y_test),le

def train_and_eval(name,clf,X_tr,y_tr,X_te,y_te,epochs=50,bs=256):
    if name in ['CNN','GRU','LSTM','RNN']:
        X_tr_3d=X_tr.reshape(X_tr.shape[0],X_tr.shape[1],1)
        X_te_3d=X_te.reshape(X_te.shape[0],X_te.shape[1],1)
        es=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
        clf.fit(X_tr_3d,y_tr,epochs=epochs,batch_size=bs,validation_split=0.1,callbacks=[es],verbose=0)
        y_prob=clf.predict(X_te_3d,verbose=0)
        y_pred=(y_prob.flatten()>0.5).astype(int) if y_prob.shape[-1]==1 else np.argmax(y_prob,axis=1)
    else:
        clf.fit(X_tr,y_tr); y_pred=clf.predict(X_te)
    avg='binary' if len(np.unique(y_te))==2 else 'weighted'
    return {'classifier':name,'accuracy':accuracy_score(y_te,y_pred),
            'precision':precision_score(y_te,y_pred,average=avg,zero_division=0),
            'recall':recall_score(y_te,y_pred,average=avg,zero_division=0),
            'f1_score':f1_score(y_te,y_pred,average=avg,zero_division=0)}

def run_all(X_tr,y_cat,X_te,y_te_cat,dim,binary=True):
    label="BINAIRE" if binary else "MULTICLASSE"
    print(f"\n{'='*60}")
    print(f"  CLASSIFICATION {label}")
    print(f"{'='*60}")
    y_tr,y_te,le=prepare_labels(y_cat,y_te_cat,binary)
    nc=2 if binary else len(np.unique(y_tr))
    print(f"  Classes: {nc}, Train: {X_tr.shape[0]}, Test: {X_te.shape[0]}")
    builders={'DT':lambda:build_dt(nc),'RF':lambda:build_rf(nc),
              'CNN':lambda:build_cnn(dim,nc),'GRU':lambda:build_gru(dim,nc),
              'LSTM':lambda:build_lstm(dim,nc),'RNN':lambda:build_rnn(dim,nc)}
    results=[]
    for name,builder in builders.items():
        print(f"  Training {name}...",end=" ")
        r=train_and_eval(name,builder(),X_tr,y_tr,X_te,y_te)
        print(f"Acc={r['accuracy']:.4f} F1={r['f1_score']:.4f}")
        results.append(r)
    return results

print("✅ Prêt")

In [ ]:
print("🔄 Baseline...")
baseline_bin=run_all(X_train_final,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,binary=True)
baseline_multi=run_all(X_train_final,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,binary=False)

print("\n🔄 AE-WGAN (Table 5 + 10% Normal)...")
enh_bin=run_all(X_augmented,y_augmented_cat,X_test_final,y_test_cat,INPUT_DIM,binary=True)
enh_multi=run_all(X_augmented,y_augmented_cat,X_test_final,y_test_cat,INPUT_DIM,binary=False)

## 7. Résultats

In [ ]:
print("\n" + "=" * 70)
print("  COMPARAISON BASELINE vs AE-WGAN (Table 5 exacte)")
print("=" * 70)
for task,br,er in [("BINAIRE",baseline_bin,enh_bin),("MULTICLASSE",baseline_multi,enh_multi)]:
    print(f"\n  --- {task} ---")
    print(f"  {'Clf':<8} {'Base Acc':>9} {'AE Acc':>9} {'Δ Acc':>7}  {'Base F1':>9} {'AE F1':>9} {'Δ F1':>7}")
    print(f"  {'─'*60}")
    for b,e in zip(br,er):
        da=e['accuracy']-b['accuracy']; df=e['f1_score']-b['f1_score']
        print(f"  {b['classifier']:<8} {b['accuracy']:>9.4f} {e['accuracy']:>9.4f} {'+'if da>=0 else ''}{da:>6.4f}  "
              f"{b['f1_score']:>9.4f} {e['f1_score']:>9.4f} {'+'if df>=0 else ''}{df:>6.4f}")

# Visualization
fig,axes=plt.subplots(2,2,figsize=(16,12))
fig.suptitle('NSL-KDD: Baseline vs AE-WGAN (Table 5)',fontsize=16,fontweight='bold')
for idx,(metric,title) in enumerate(zip(['accuracy','precision','recall','f1_score'],['Accuracy','Precision','Recall','F1 Score'])):
    ax=axes[idx//2][idx%2]
    clfs=[r['classifier'] for r in baseline_bin]; x=np.arange(len(clfs)); w=0.2
    ax.bar(x-1.5*w,[r[metric] for r in baseline_bin],w,label='Baseline Bin',color='#3498db',alpha=0.8)
    ax.bar(x-0.5*w,[r[metric] for r in enh_bin],w,label='AE-WGAN Bin',color='#e74c3c',alpha=0.8)
    ax.bar(x+0.5*w,[r[metric] for r in baseline_multi],w,label='Baseline Multi',color='#2ecc71',alpha=0.8)
    ax.bar(x+1.5*w,[r[metric] for r in enh_multi],w,label='AE-WGAN Multi',color='#f39c12',alpha=0.8)
    ax.set_xlabel('Classifieur');ax.set_ylabel(title);ax.set_title(title)
    ax.set_xticks(x);ax.set_xticklabels(clfs);ax.legend(fontsize=8);ax.set_ylim(0.5,1.05);ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('results_table5.png',dpi=150,bbox_inches='tight')
plt.show()

# CSV + Drive
all_results=[]
for r in baseline_bin: all_results.append({**r,'type':'Baseline','classification':'Binary'})
for r in enh_bin: all_results.append({**r,'type':'AE-WGAN','classification':'Binary'})
for r in baseline_multi: all_results.append({**r,'type':'Baseline','classification':'Multiclass'})
for r in enh_multi: all_results.append({**r,'type':'AE-WGAN','classification':'Multiclass'})
df_res=pd.DataFrame(all_results)
df_res.to_csv('results_table5.csv',index=False)
import shutil
shutil.copy('results_table5.csv','/content/drive/MyDrive/')
shutil.copy('results_table5.png','/content/drive/MyDrive/')
print("\n✅ Résultats sauvegardés et copiés sur Drive")
print(df_res.to_string(index=False))